In [1]:
archivo = "Jupyter/siglas_CIA/ControlSiglas_CIA_Interno.xlsx"

In [39]:
import psycopg2

try:
    conn = psycopg2.connect(
        dbname="dara_mallas_18",
        user="odoo",
        password="root123",
        host="localhost",
        port="5432"
    )
    
    print("✅ Conexión exitosa a la base de datos")

    # Validación real (recomendado)
    cursor = conn.cursor()
    cursor.execute("SELECT 1")
    
    print("✅ Query de prueba ejecutada correctamente")

except Exception as e:
    print("❌ Error de conexión:")
    print(e)

✅ Conexión exitosa a la base de datos
✅ Query de prueba ejecutada correctamente


Traigo cursos existentes

In [2]:
import pandas as pd

df = pd.read_excel(f"/home/odoo/Documentos/DESARROLLO/{archivo}", sheet_name="Hoja1")

f_sql = open('./siglas_CIA/1_siglas_limpia_sin_where','w')

for i, item in df.iterrows():
    valor = str(item['SCBCRSE_CODE']).strip()

    if len(valor) < 8:
        print(f"Fila {i} inválida: '{valor}'")
        continue

    sigla = valor[:4]
    code = valor[4:8]

    name = str(item['Nombre Largo']).replace("'", "''")
    short = str(item['Nombre Corto']).replace("'", "''")

    # ✅ INSERT válido
    sql = f"""
        insert into dara_mallas_subject
        (name, name_en, short_name, code, course_number, subject_name_id, new_subject, company_id, book_check)
        select
            '{name}',
            NULL,
            '{short}',
            '{valor}',
            '{code}',
            (select id from dara_mallas_subject_name
            where code = '{sigla}' and company_id = 4),
            True,
            4,
            True
        ;\n
        
        insert into dara_mallas_subject_class
        (subject_class_id, subject_id, company_id)
        values(
            (select id from dara_mallas_subject_class_name
            where name = 'Carrera' and company_id = 4),
            (select id from dara_mallas_subject
            where code = '{valor}' and company_id = 4
            limit 1),
            4
        );\n
        """

    f_sql.write(sql)

In [ ]:
import pandas as pd

df = pd.read_excel(f"/home/odoo/Documentos/DESARROLLO/{archivo}", sheet_name="Hoja1")

f_sql = open('./siglas_CIA/1_siglas_limpia','w')

for i, item in df.iterrows():
    valor = str(item['SCBCRSE_CODE']).strip()

    if len(valor) < 8:
        print(f"Fila {i} inválida: '{valor}'")
        continue

    sigla = valor[:4]
    code = valor[4:8]

    name = str(item['Nombre Largo']).replace("'", "''")
    short = str(item['Nombre Corto']).replace("'", "''")

    # ✅ INSERT válido
    sql = f"""
        insert into dara_mallas_subject
        (name, name_en, short_name, code, course_number, subject_name_id, new_subject, company_id, book_check)
        select
            '{name}',
            NULL,
            '{short}',
            '{valor}',
            '{code}',
            (select id from dara_mallas_subject_name
            where code = '{sigla}' and company_id = 4),
            True,
            4,
            True
        where not exists (
            select 1 from dara_mallas_subject
            where code = '{valor}' and company_id = 4
        );\n
        """

    f_sql.write(sql)

In [40]:
cursor.execute("SELECT code FROM dara_mallas_subject where company_id = 4")
codes_bd = set(row[0] for row in cursor.fetchall())

In [41]:
cursor.execute("""
    SELECT code FROM dara_mallas_subject_name
    WHERE company_id = 4
""")

subject_names_bd = set(row[0] for row in cursor.fetchall())

In [42]:
import pandas as pd

df = pd.read_excel(f"/home/odoo/Documentos/DESARROLLO/{archivo}", sheet_name="Hoja1")

f_sql = open('./siglas_CIA/1_siglas.sql','w')
f_dup = open('./siglas_CIA/duplicados.txt','w')

for i, item in df.iterrows():
    valor = str(item['SCBCRSE_CODE']).strip()

    if len(valor) < 8:
        print(f"Fila {i} inválida: '{valor}'")
        continue

    sigla = valor[:4]
    code = valor[4:8]

    name = str(item['Nombre Largo']).replace("'", "''")
    short = str(item['Nombre Corto']).replace("'", "''")

    # 🔴 VALIDACIÓN 1: ya existe en subject
    if valor in codes_bd:
        f_dup.write(f"{valor} -> NO CREADO (ya existe en dara_mallas_subject)\n")
        continue

    # 🔴 VALIDACIÓN 2: sigla no existe en subject_name
    if sigla not in subject_names_bd:
        f_dup.write(f"{valor} -> NO CREADO (sigla {sigla} NO existe en subject_name)\n")
        continue

    # ✅ INSERT válido
    sql = f"""
        insert into dara_mallas_subject
        (name, name_en, short_name, code, course_number, subject_name_id, new_subject, company_id, book_check)
        select
            '{name}',
            NULL,
            '{short}',
            '{valor}',
            '{code}',
            (select id from dara_mallas_subject_name
            where code = '{sigla}' and company_id = 4),
            True,
            4,
            True
        where not exists (
            select 1
            from dara_mallas_subject
            where code = '{valor}' and company_id = 4
        );\n
        """

    f_sql.write(sql)